In [1]:
import numpy as np
import pandas as pd
import warnings

# Biogeme core
import biogeme.biogeme as bio
from biogeme import models
from biogeme.expressions import Beta, Variable, bioDraws

import biogeme.database as db

warnings.filterwarnings("ignore")

C:\Users\vladimir.koev\AppData\Local\Programs\Python\Python311\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
C:\Users\vladimir.koev\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  fr

# Phase 4 — Multinomial Logit (MNL) Model Estimation

## 4.0 Overview

This phase estimates commuter mode choice models separately for Amsterdam and London using the Random Utility Maximisation (RUM) framework and the Multinomial Logit (MNL) specification (McFadden, 1974). The dependent variable is the chosen commute mode — **car** (reference), **public transport**, or **bicycle** — and estimation proceeds by maximum likelihood via Biogeme (Bierlaire, 2020).

We follow the alternative-specific coefficient approach recommended by Ben-Akiva & Lerman (1985): socioeconomic attributes (income, car ownership, driving licence, age, gender) enter the utility functions of PT and bicycle *relative to car*, whose coefficients are normalised to zero for identification. Travel time remains a generic coefficient, as it represents level-of-service and carries the same behavioural interpretation across modes. Two mode-specific variables are added based on theoretical priors: **number of transfers** (PT only — each transfer adds disutility; Ortúzar & Willumsen, 2011) and **trip distance** (bicycle only — physical effort increases with distance).

Per city we estimate two nested specifications: **Experiment A** (baseline) and **Experiment B** (adds peak-hour effects). The final model per city is selected via a likelihood-ratio test, McFadden's pseudo-$R^2$, and coefficient stability. Results feed directly into hypothesis testing in Phase 5.

## 4.1 Data Preparation for MNL

The first step is to create model-ready datasets for each city with identical coding rules. One row per observed choice, numerical coding of alternatives, clean covariate ranges, and no missing values in model variables. This step also enforces the same dependent variable mapping across cities, so estimated coefficients are directly comparable.

The code below constructs Amsterdam and London worksets from the harmonised cleaned datasets, applies defensible variable typing, and creates a shared alternative coding: car = 1, pt = 2, bike = 3.

In [2]:
amst_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_processed.csv")

In [3]:
lnd_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_processed.csv")

In [4]:
pd.set_option('display.max_columns', None)
lnd_data.head(10)

,person_id,trip_id,trip_purpose,mode_detailed,travel_time_min,distance_km,departure_hour,weekday,is_holiday,age_band,gender,income_quintile,has_driving_license,n_cars_household,weight_trip,weight_person,city,is_peak,chosen_mode,n_transfers,has_transfer,n_legs
0,2023009631,2023109722,1,10,60.0,11.265408,0,3,4,8,1,1,0,0,0.986298,0.916839,London,0,pt,0,0,1
1,2023007220,2023083399,1,10,40.0,17.702784,23,5,3,6,2,5,0,3,1.032084,1.143973,London,0,pt,0,0,1
2,2023000996,2023011523,1,10,15.0,3.379622,0,1,3,6,1,3,0,0,1.461324,1.431693,London,0,pt,0,0,1
3,2023008133,2023092873,1,10,47.0,6.759245,-1,5,4,5,2,2,0,1,1.396613,1.308343,London,0,pt,2,1,3
4,2023014646,2023170543,1,10,60.0,24.944832,-1,3,3,5,2,5,1,4,0.691940,0.698998,London,0,pt,1,1,2
5,2023002887,2023033439,1,2,30.0,8.851392,5,4,4,5,1,4,0,1,1.678506,1.286774,London,0,bike,0,0,1
6,2023000323,2023003673,1,7,35.0,24.140160,5,1,3,8,2,5,1,2,1.073553,1.142800,London,0,pt,1,1,2
7,2023010282,2023118043,1,3,31.0,20.921472,5,2,3,8,1,2,1,2,0.537043,0.587190,London,0,car,0,0,1
8,2023011915,2023138347,1,7,45.0,5.954573,6,2,4,6,1,2,1,1,1.564170,1.254609,London,1,pt,0,0,1
9,2023014242,2023165626,1,10,20.0,3.218688,6,4,3,5,1,5,1,0,1.481708,1.272565,London,1,pt,0,0,1


In [5]:
mode_to_id = {"car": 1, "pt": 2, "bike": 3}

# MNL Variables
model_columns = [
    "chosen_mode",
    "travel_time_min",
    "distance_km",         
    "n_transfers",           
    "income_quintile",
    "has_driving_license",
    "n_cars_household",
    "age_band",
    "gender",
    "is_peak",
    "weight_trip",
]

In [8]:
def build_city_workset(city_data, city_name):
    
    ws = city_data[model_columns].copy()

    # 1. Drop rows with any missing value in model columns
    # n_before = len(ws)
    # ws = ws.dropna(subset=model_columns)
    # print(f"{city_name}: dropped {n_before - len(ws):,} rows with missing model vars")

    # 2. Encode dependent variable as integer
    ws["choice_code"] = ws.chosen_mode.map(mode_to_id).astype(int)

    # 3. Cast all numeric columns explicitly
    numeric_cols = [
        "travel_time_min", "distance_km", "n_transfers",
        "income_quintile", "has_driving_license",
        "n_cars_household", "age_band", "gender",
        "is_peak", "weight_trip", "choice_code",
    ]
    ws[numeric_cols] = ws[numeric_cols].apply(pd.to_numeric, errors = "coerce")
    ws = ws.dropna(subset = numeric_cols)

    # 4. Keep only final modelling columns
    keep = ["choice_code"] + [c for c in numeric_cols if c != "choice_code"]
    ws = ws[keep].reset_index(drop=True)

    # 5. Summary
    print(f"{city_name}: {ws.shape[0]:,} rows ready for MNL")
    return ws


# --- Build worksets ---
amsterdam_workset = build_city_workset(amst_data, "Amsterdam")
london_workset    = build_city_workset(lnd_data, "London")

Amsterdam: 681 rows ready for MNL
London: 3,315 rows ready for MNL


In [9]:
amsterdam_workset.columns == london_workset.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True])

In [10]:
amsterdam_workset.head(5)

,choice_code,travel_time_min,distance_km,n_transfers,income_quintile,has_driving_license,n_cars_household,age_band,gender,is_peak,weight_trip
0,2,50.0,10.2,1,4,1.0,0.0,7,2.0,1,221250.406070
1,2,45.0,11.0,0,4,1.0,0.0,7,2.0,1,221250.406070
2,1,20.0,13.2,0,2,1.0,3.0,9,1.0,0,133191.103713
3,1,25.0,13.2,0,2,1.0,3.0,9,1.0,0,133191.103713
4,2,35.0,9.3,2,1,0.0,0.0,5,2.0,0,112778.553061


## 4.2 MNL Specification and Estimation Function

The utility of alternative $j$ for individual $n$ follows the RUM decomposition $U_{jn} = V_{jn} + \varepsilon_{jn}$, where the error terms $\varepsilon$ are i.i.d. Type-I Extreme Value, yielding the MNL choice probability (Train, 2009, Ch. 3):

$$P(i \mid C_n) = \frac{e^{V_{in}}}{\sum_{j \in C_n} e^{V_{jn}}}$$

**Identification.** With $J=3$ alternatives and individual-specific (not alternative-specific) regressors, we normalise all car coefficients to zero. The estimated parameters then measure the effect of each variable on the utility of PT or bicycle *relative to car* (Ben-Akiva & Lerman, 1985, §5.2).

**Specification.** Travel time (`b_time`) is generic — its marginal disutility is assumed equal across modes, which is standard when no alternative-specific LoS data are available. All socioeconomic variables receive alternative-specific coefficients. Two theoretically motivated LoS variables are added: `b_transfers` in $V_{\text{PT}}$ (each transfer raises generalised cost; Wardman, 2004) and `b_dist_bike` in $V_{\text{bike}}$ (cycling effort increases with distance). In Experiment B, `is_peak` enters as an alternative-specific variable to test whether peak-hour commuting shifts mode preference.

### 4.2 Model Specification and Statistical Tests

In this step we are going to estimate two specifications per city as per below.

1. Experiment A (baseline): includes travel time and socioeconomic variables.
2. Experiment B (robustness): adds peak-hour effects. The utility functions use alternative-specific constants (ASC), with car as the reference alternative (${ASC}_{car}=0$) for identification. In discrete choice models, we only care about the difference in utility between alternatives, not the total utility.

It is important to mention what ASC is. Generally, an ASC represents the average impact of all unobserved factors on the utility of a specific mode. For example,  the "Hidden" factors in our model. Our model includes travel time but it doesn't include variables like "comfort," "prestige," "privacy," or "safety".  If travel time and cost were exactly the same for all modes, would people still prefer one over the other? The ASC captures that inherent bias.

For model comparison, we are going to calculate the differences in log-likelihood and pseudo-$R^2$, and inspect coefficient sign/significance stability. For each parameter $\beta_k$, the hypothesis test is:

$$
H_0: \beta_k = 0 \quad \text{vs} \quad H_1: \beta_k \neq 0
$$

at $\alpha = 0.05$. A p-value below $\alpha$ implies rejection of $H_0$, indicating statistical evidence that the covariate is associated with mode choice.

In [26]:
def estimate_city_mnl(city_workset, city_name, include_peak=False):

    tag = "exp_b_peak" if include_peak else "exp_a_base"
    database = db.Database(f"{city_name.lower()}_{tag}", city_workset)

    # --- Biogeme variable handles ---
    choice = Variable("choice_code")
    travel_time_min = Variable("travel_time_min")
    distance_km = Variable("distance_km")
    n_transfers = Variable("n_transfers")
    income_quintile = Variable("income_quintile")
    has_driving_lic = Variable("has_driving_license")
    n_cars = Variable("n_cars_household")
    age_band = Variable("age_band")
    gender = Variable("gender")
    is_peak = Variable("is_peak")

    # ASC_car = 0)
    asc_pt = Beta("asc_pt",   0, None, None, 0)
    asc_bike = Beta("asc_bike", 0, None, None, 0)

    # travel-time coefficient
    b_time = Beta("b_time", -0.05, None, None, 0)

    # Alternative-specific socioeconomic (car = 0)
    b_income_pt = Beta("b_income_pt",   0, None, None, 0)
    b_income_bike = Beta("b_income_bike", 0, None, None, 0)

    b_license_pt = Beta("b_license_pt",   0, None, None, 0)
    b_license_bike = Beta("b_license_bike", 0, None, None, 0)

    b_cars_pt = Beta("b_cars_pt",   0, None, None, 0)
    b_cars_bike = Beta("b_cars_bike", 0, None, None, 0)

    b_age_pt = Beta("b_age_pt",   0, None, None, 0)
    b_age_bike = Beta("b_age_bike", 0, None, None, 0)

    b_gender_pt = Beta("b_gender_pt",   0, None, None, 0)
    b_gender_bike = Beta("b_gender_bike", 0, None, None, 0)

    # Mode-specific LoS variables
    b_transfers = Beta("b_transfers", 0, None, None, 0)   # PT only
    b_dist_bike = Beta("b_dist_bike", 0, None, None, 0)   # bike only

    # Optional peak-hour (Experiment B) — alternative-specific ===
    if include_peak:
        b_peak_pt = Beta("b_peak_pt",   0, None, None, 0)
        b_peak_bike = Beta("b_peak_bike", 0, None, None, 0)
    else:
        b_peak_pt = 0
        b_peak_bike = 0

    # Utility functions:
    
    # V_car: reference — only generic time enters
    v_car = b_time * travel_time_min

    # V_pt: ASC + generic time + socioeconomic (alt-specific) + transfers
    v_pt = (
        asc_pt
        + b_time * travel_time_min
        + b_income_pt * income_quintile
        + b_license_pt * has_driving_lic
        + b_cars_pt * n_cars
        + b_age_pt * age_band
        + b_gender_pt * gender
        + b_transfers * n_transfers
        + b_peak_pt * is_peak
    )

    # V_bike: ASC + generic time + socioeconomic (alt-specific) + distance
    v_bike = (
        asc_bike
        + b_time * travel_time_min
        + b_income_bike * income_quintile
        + b_license_bike * has_driving_lic
        + b_cars_bike * n_cars
        + b_age_bike * age_band
        + b_gender_bike * gender
        + b_dist_bike * distance_km
        + b_peak_bike * is_peak
    )

    # --- Choice model ---
    V  = {1: v_car, 2: v_pt, 3: v_bike}
    av = {1: 1, 2: 1, 3: 1}          # all modes available
    logprob = models.loglogit(V, av, choice)

    # --- Biogeme estimation ---
    biogeme_model = bio.BIOGEME(database, logprob, generate_html=False)
    biogeme_model.modelName = f"mnl_{city_name.lower()}_{tag}"
    
    results = biogeme_model.estimate()

    # --- Extract results ---
    beta_table = results.get_estimated_parameters()
    beta_table = beta_table.reset_index().rename(columns={"index": "parameter"})
    beta_table["city"] = city_name
    beta_table["spec"] = tag
    
    # --- Fit statistics ---
    n_obs = city_workset.shape[0]
    ll_null = -n_obs * np.log(3)

    general_stats = results.get_general_statistics()

    # Extract LL and n_params — values may be tuples like (value, precision_str)
    ll_star = None
    n_params = None
    for key, val in general_stats.items():
        key_str = str(key).lower()
        # val is often a tuple: (numeric_value, format_string)
        numeric_val = val[0] if isinstance(val, (tuple, list)) else val
        if 'final log likelihood' in key_str:
            ll_star = float(numeric_val)
        if 'number of estimated parameters' in key_str:
            n_params = int(numeric_val)

    # Fallback
    if ll_star is None:
        try:
            ll_star = float(results.data.logLike)
        except Exception:
            ll_star = np.nan
    if n_params is None:
        n_params = len(beta_table)

    rho2 = 1.0 - (ll_star / ll_null)

    fit_stats = {
        "city": city_name, "spec": tag,
        "n_obs": n_obs, "n_params": n_params,
        "LL_null": round(ll_null, 2),
        "LL_star": round(ll_star, 2),
        "rho2": round(rho2, 4),
    }

    # --- Print results (auto-detect column names) ---
    print(f"\n{'='*60}")
    print(f"✓ {city_name} | {tag} | n={n_obs:,} | LL*={ll_star:.2f} | ρ²={rho2:.4f}")
    print(f"{'='*60}")
    print(beta_table.to_string(index=False))

    return biogeme_model, results, beta_table, fit_stats

## 4.3 Amsterdam — Model Estimation

We now estimate the two specifications for Amsterdam (n=681). Given the relatively small sample, we expect wider confidence intervals and potentially insignificant coefficients for some socioeconomic variables. Amsterdam's mode split is bicycle-dominant (60%), which differs sharply from London and should be reflected in the ASC estimates. Experiment A provides the baseline; Experiment B adds peak-hour alternative-specific effects to test whether morning/evening rush shifts mode preference for Amsterdam commuters.

In [27]:
amst_model_a, amst_results_a, amst_beta_a, amst_fit_a = estimate_city_mnl(
    amsterdam_workset, "Amsterdam", include_peak = False
)

amst_model_b, amst_results_b, amst_beta_b, amst_fit_b = estimate_city_mnl(
    amsterdam_workset, "Amsterdam", include_peak=True
)


✓ Amsterdam | exp_a_base | n=681 | LL*=-208.90 | ρ²=0.7208
 parameter           Name      Value  Robust std err.  Robust t-stat.  Robust p-value      city       spec
         0         b_time  -0.050000         0.000000   1.797693e+308    0.000000e+00 Amsterdam exp_a_base
         1         asc_pt   6.951771         1.846433    3.764974e+00    1.665661e-04 Amsterdam exp_a_base
         2    b_income_pt   0.011611         0.167639    6.926304e-02    9.447802e-01 Amsterdam exp_a_base
         3   b_license_pt -11.189859         0.639107   -1.750857e+01    0.000000e+00 Amsterdam exp_a_base
         4      b_cars_pt  -1.667400         0.548825   -3.038127e+00    2.380539e-03 Amsterdam exp_a_base
         5       b_age_pt   0.342095         0.199328    1.716240e+00    8.611818e-02 Amsterdam exp_a_base
         6    b_gender_pt   0.539482         0.520898    1.035677e+00    3.003529e-01 Amsterdam exp_a_base
         7    b_transfers  16.536646         0.676174    2.445621e+01    0.000000e+0

## 4.4 London — Model Estimation

London's sample (n=3,394) is roughly five times larger than Amsterdam's, providing substantially more statistical power. The mode split is PT-dominant (53%), with car at 39% and bicycle at only 8%. We expect stronger significance across all coefficients and tighter confidence intervals. The same two specifications are estimated, preserving strict comparability with Amsterdam.

In [28]:
lnd_model_a, lnd_results_a, lnd_beta_a, lnd_fit_a = estimate_city_mnl(
    london_workset, "London", include_peak=False
)

lnd_model_b, lnd_results_b, lnd_beta_b, lnd_fit_b = estimate_city_mnl(
    london_workset, "London", include_peak=True
)


✓ London | exp_a_base | n=3,315 | LL*=-1646.32 | ρ²=0.5479
 parameter           Name     Value  Robust std err.  Robust t-stat.  Robust p-value   city       spec
         0         b_time -0.050000         0.000000   1.797693e+308    0.000000e+00 London exp_a_base
         1         asc_pt  8.769030         0.760422    1.153179e+01    0.000000e+00 London exp_a_base
         2    b_income_pt  0.323403         0.040515    7.982303e+00    1.332268e-15 London exp_a_base
         3   b_license_pt -5.844506         0.537686   -1.086975e+01    0.000000e+00 London exp_a_base
         4      b_cars_pt -1.830207         0.117222   -1.561314e+01    0.000000e+00 London exp_a_base
         5       b_age_pt -0.482765         0.049531   -9.746805e+00    0.000000e+00 London exp_a_base
         6    b_gender_pt  0.407737         0.112994    3.608498e+00    3.079753e-04 London exp_a_base
         7    b_transfers 14.514221         0.294305    4.931693e+01    0.000000e+00 London exp_a_base
         8   